# Problem 039:  Integer Right Triangles

> If $p$ is the perimeter of a right angle triangle with integral length sides, $\{a, b, c\}$, there are exactly three solutions for $p = 120$.
>
> $\{20,48,52\}$, $\{24,45,51\}$, $\{30,40,50\}$
>
> For which value of $p \le 1000$, is the number of solutions maximised?

## Naive Solution $\mathcal O\left(N^2\right)$

The naive approach is to search for Pythagoarean triplets by searching all leg lengths and seeing if the resulting hypotenuse is an integer.  Then, a histogram of perimeters found in the search is generated.  The index of the maximum value is returned as the answer.

I try to avoid any computations that require floating point precision.  To determine if the hypotenuse of a right triangle is an integer is to check if a number is a perfect square.  One approach is to take the squareroot of the number as a `float`, cast it to an `int`, and see if the square equals the number again.  For the numbers considered here, this would work fine.  But at some point the rounding error in the squareroot calculation will lead to inexact calculations and create unreliable results.  Instead, I make a dictionary of squares with their roots and just check if the number is in the dictionary.  One could point out that this approach would be inefficient in the case when the rounding error of a `float` is of concern, so be it.  I still avoid dealing with floating point values.  I don't think it's worth diving into Heron's algorithm at this point to handle more complicated situations.

This approach will take $\mathcal O(N^2)$ to complete since all combination of leg values are considered.

In [35]:
%%time

def p039_naive(N: int) -> int:
    cnt = [0] * (N+1)
    
    sqrs = {i**2:i for i in range(N+1)}

    for i in range(1, N // 2):
        jmax = (N * (N - 2 * i)) // (2 * (N - i)) + 1
        for j in range(1, jmax):
            k = i**2 + j**2
            if k in sqrs:
                k = sqrs[k]
                if i + j + k <= N:
                    cnt[i+j+k] += 1

    return cnt.index(max(cnt))
    
N = 1_000
ans = p039_naive(N)

print(ans)

840
CPU times: user 67.7 ms, sys: 1 ms, total: 68.7 ms
Wall time: 68.1 ms


## Faster Solution $\mathcal O\left(N\ln N \right)$

There are direct ways of generating all Pythagorean triplets to avoid having to search through all possible combinations of values.  I have already discussed one way of doing this in Problem 009, consisting of generating coprime pairs of numbers.  Another way, as outlined in the Wolfram MathWorld page for [Pythagorean Triples](https://mathworld.wolfram.com/PythagoreanTriple.html) is to use the following three transformation matrices:
$$A_0 = \begin{bmatrix} 1 & -2 & 2 \\ 2 & -1 & 2 \\ 2 & -2 & 3 \end{bmatrix}; A_1 = \begin{bmatrix} 1 & 2 & 2 \\ 2 & 1 & 2 \\ 2 & 2 & 3 \end{bmatrix}; A_2 = \begin{bmatrix} -1 & 2 & 2 \\ -2 & 1 & 2 \\ -2 & 2 & 3 \end{bmatrix}.$$
From these three matrices, all primative Pythagorean triplets can be generated by repeated applications of these matrices on the vector $(3,4,5)$.  Given a primitive triplet, all triplets can be calculated by multiplying the primitive triplet by a scalar.  One nice aspect about this approach is that each successive application of the transformation matrices generates a triplet where each side of the triangle is greater than the corresponding side in the previous triplet.  The code below is a recursive generator that generates triplets until the target perimeter is exceded.

The approach is the same as above besides the generation of Pythogorean triplets.  Namely, a histogram is generated for all perimeters less than or equal to $N$ and the maximum is found after searching all triplets.

As for time scaling, there are $\sim N$ primitive triangles with a perimeter less than $N$, and $\sim N\ln N$ total triangles, making the overall calculation $\mathcal O(N\ln N)$.

In [34]:
%%time

def pythagoras_generator(a: int, b: int, c:int, N: int) -> (int, int, int):
    if a + b + c <= N:
        yield a, b, c
        ap =     a - 2 * b + 2 * c
        bp = 2 * a -     b + 2 * c
        cp = 2 * a - 2 * b + 3 * c
        yield from pythagoras_generator(ap, bp, cp, N)
        ap =     a + 2 * b + 2 * c
        bp = 2 * a +     b + 2 * c
        cp = 2 * a + 2 * b + 3 * c
        yield from pythagoras_generator(ap, bp, cp, N)
        ap = -     a + 2 * b + 2 * c
        bp = - 2 * a +     b + 2 * c
        cp = - 2 * a + 2 * b + 3 * c
        yield from pythagoras_generator(ap, bp, cp, N)
        


def p039(N: int) -> int:
    cnt = [0] * (N+1)

    for sides in pythagoras_generator(3,4,5,N):
        p = sum(sides)
        cnt[p::p] = [a + 1 for a in cnt[p::p]]

    return cnt.index(max(cnt))
    
N = 1_000
ans = p039(N)


print(ans)

840
CPU times: user 508 μs, sys: 5 μs, total: 513 μs
Wall time: 506 μs
